# run14 A/B: v1 vs v2 on the Kaggle A100

Same devset, same model instance, sequential. Writes results/results_v{1,2}/progress.jsonl and an ab_summary.

In [ ]:
import os
os.environ.update({
    'TRANSFORMERS_NO_TF': '1', 'TRANSFORMERS_NO_FLAX': '1', 'TRANSFORMERS_OFFLINE': '1',
    'VLLM_NO_USAGE_STATS': '1', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
    'VLLM_ENGINE_READY_TIMEOUT_S': '1800', 'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS': '1',
    'NCCL_DEBUG': 'WARN', 'LITELLM_LOCAL_MODEL_COST_MAP': 'True', 'LITELLM_TELEMETRY': 'False',
    'LITELLM_LOG': 'WARNING', 'OPENAI_API_KEY': 'EMPTY', 'LOCAL_API_KEY': 'EMPTY',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'OTEL_SDK_DISABLED': 'true', 'OTEL_PYTHON_DISABLED': 'true',
})
import logging
for name in ('LiteLLM', 'litellm', 'httpx', 'httpcore', 'openai', 'urllib3', 'opentelemetry'):
    lg = logging.getLogger(name)
    lg.setLevel(logging.WARNING)
    if 'litellm' in name.lower():
        lg.propagate = False
        lg.handlers.clear()
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'gpus', torch.cuda.device_count())


In [ ]:
import glob, os, shutil, subprocess, sys
from pathlib import Path

for pth_pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
                    '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

WH = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
tmp = Path('/tmp/wheelhouse'); tmp.mkdir(exist_ok=True)
n = 0
for w in sorted(WH.glob('*.whl')):
    if 'cutlass' in w.name.lower():
        continue
    target_name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    target = tmp / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)
    n += 1
print('installing', n, 'wheels')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall',
                *sorted(str(w) for w in tmp.glob('*.whl'))], check=True)
import importlib
importlib.invalidate_caches()
import vllm, transformers
print('vllm', vllm.__version__, 'transformers', transformers.__version__)


In [ ]:
import base64, glob
from pathlib import Path
COMP = Path(glob.glob('/kaggle/input/**/tasks.jsonl', recursive=True)[0]).parent
assert (COMP / 'docker' / 'imp.py').exists()
Path('/kaggle/working').mkdir(parents=True, exist_ok=True)
EVAL_SRC = base64.b64decode(''.join(r'''
IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJMb2NhbCBzY29yZXItbGlrZSBldmFsdWF0aW9uIG9mIGEgc3VibWlzc2lvbiBhZ2VudCBvbiB0aGUgcHVibGljIHRhc2tzLgoKVXNhZ2U6CiAgdmVudi1ldmFsL2Jpbi9weXRob24gc2NyaXB0cy9ldmFsX2xvY2FsLnB5IC0tdGFza3MgMCwxLDIgLS1zaGFyZCAwIC0tbnVtLXNoYXJkcyAxCkVudjoKICBNT0RFTF9VUkwgIHZMTE0gT3BlbkFJIGJhc2UgdXJsIChkZWZhdWx0IGh0dHA6Ly8xMjcuMC4wLjE6ODAwMC92MSkKICBNT0RFTF9OQU1FIChkZWZhdWx0IGdlbW1hLTQtMzFiLWl0LXFhdC13NGExNi1jdCkKICBBR0VOVF9ESVIgIChkZWZhdWx0IGFnZW50KQpSdW5zIHRoZSBhZ2VudCBwaGFzZSAoUGhhc2UgMSkgKyB2ZXJpZmljYXRpb24gKFBoYXNlIDIpIGluc2lkZSBzdWJwcm9jZXNzCnNhbmRib3hlcyB3aG9zZSB2ZW52L3B5dGhvbiBhbmQgaW5zdGFsbGVkIHdoZWVscyBtaXJyb3IgdGhlIHNjb3JlciBpbWFnZS4KIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgYXN5bmNpbwppbXBvcnQgZ2xvYgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJlCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGFyZmlsZQppbXBvcnQgdGltZQppbXBvcnQgdG9tbGxpYgppbXBvcnQgemlwZmlsZQpmcm9tIGNvbmN1cnJlbnQuZnV0dXJlcyBpbXBvcnQgVGhyZWFkUG9vbEV4ZWN1dG9yCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKUlVOID0gUGF0aCgnL2thZ2dsZS93b3JraW5nJykKQ09NUCA9IFBhdGgoJy9rYWdnbGUvaW5wdXQvY29tcGV0aXRpb25zL2dlbW1hLTQtZGV2ZWxvcGVyLWFnZW50JykKV0hFRUxIT1VTRSA9IFBhdGgoJy9rYWdnbGUvaW5wdXQvZGF0YXNldHMvbWV0cmljL2dlbW1hLTQtZGV2ZWxvcGVyLWFnZW50LXdoZWVsaG91c2UnKQpFWFRSQSA9IFBhdGgoZ2xvYi5nbG9iKCcva2FnZ2xlL2lucHV0LyoqL2lubGluZV9zbmFwc2hvdC0qLndobCcsIHJlY3Vyc2l2ZT1UcnVlKVswXSkucGFyZW50ClBZRFMgPSBQYXRoKGdsb2IuZ2xvYignL2thZ2dsZS9pbnB1dC8qKi9jcHl0aG9uLTMuMTItbGludXgudGd6LmJpbicsIHJlY3Vyc2l2ZT1UcnVlKVswXSkucGFyZW50CgppbXBvcnQgaW1wb3J0bGliCmltcG9ydCBsb2dnaW5nCmltcG9ydCByZQppbXBvcnQgdG9tbGxpYgppbXBvcnQgemlwZmlsZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmZyb20gcGFja2FnaW5nLnJlcXVpcmVtZW50cyBpbXBvcnQgUmVxdWlyZW1lbnQKZnJvbSBwYWNrYWdpbmcudmVyc2lvbiBpbXBvcnQgVmVyc2lvbgpmcm9tIHN3ZWdlbW1hLmNvbmZpZyBpbXBvcnQgRXZhbENvbmZpZywgYnVpbGRfc3VibWlzc2lvbl9saW1pdHMKZnJvbSBzd2VnZW1tYS5ldmFsdWF0ZSBpbXBvcnQgRXZhbHVhdG9yCmZyb20gc3dlZ2VtbWEubW9kZWxzIGltcG9ydCBsb2FkX3Rhc2tzCmZyb20gc3dlZ2VtbWEuc2FuZGJveC5zdWJwcm9jZXNzIGltcG9ydCBTdWJwcm9jZXNzTWFuYWdlcgppbXBvcnQgc3dlZ2VtbWEuaGFybmVzcy52ZXJpZmljYXRpb24gYXMgdmVyaWZpY2F0aW9uCmltcG9ydCBzd2VnZW1tYS5oYXJuZXNzLmFnZW50X3J1bm5lciBhcyBhZ2VudF9ydW5uZXIKZnJvbSBhZGtfc3VibWlzc2lvbiBpbXBvcnQgTW9kZWxSZWdpc3RyeQpmcm9tIGdvb2dsZS5hZGsubW9kZWxzLmxpdGVfbGxtIGltcG9ydCBMaXRlTGxtCmZyb20gZ29vZ2xlLmFkay5hZ2VudHMuY29udGV4dF9jYWNoZV9jb25maWcgaW1wb3J0IENvbnRleHRDYWNoZUNvbmZpZwpmcm9tIGdvb2dsZS5hZGsuYXBwcy5fY29uZmlncyBpbXBvcnQgRXZlbnRzQ29tcGFjdGlvbkNvbmZpZwoKbG9nZ2luZy5nZXRMb2dnZXIoKS5zZXRMZXZlbChsb2dnaW5nLkVSUk9SKQoKbm9ybSA9IGxhbWJkYSBuOiByZS5zdWIocidbLV8uXSsnLCAnXycsIG4pLmxvd2VyKCkKCgpkZWYgZml0cyh3OiBzdHIsIHRhZzogc3RyKSAtPiBib29sOgogICAgbSA9IHJlLmZ1bGxtYXRjaChyJyguKyktKFteLV0rKS0oW14tXSspLSguKylcLndobCcsIFBhdGgodykubmFtZSkKICAgIGlmIG5vdCBtOgogICAgICAgIHJldHVybiBUcnVlCiAgICBweSwgYWJpLCBwbGF0ID0gbS5ncm91cCgyKSwgbS5ncm91cCgzKSwgbS5ncm91cCg0KQogICAgaWYgcGxhdCBub3QgaW4gKCdhbnknLCkgYW5kICdsaW51eCcgbm90IGluIHBsYXQ6CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICBpZiBhYmkgPT0gJ2FiaTMnOgogICAgICAgIHJldHVybiBpbnQocmUuc3ViKHInXEQnLCAnJywgcHkpIG9yIDApIDw9IGludCh0YWdbMjpdKQogICAgcmV0dXJuIGFueSh0LnN0YXJ0c3dpdGgoJ3B5MycpIGZvciB0IGluIHB5LnNwbGl0KCcuJykpIG9yIHRhZyBpbiBweS5zcGxpdCgnLicpCgoKV0hFRUxTID0ge30KZm9yIHcgaW4gWyooQ09NUCAvICd3aGVlbHMnKS5nbG9iKCcqLndobCcpLCAqRVhUUkEuZ2xvYignKi53aGwnKSwgKlBZRFMuZ2xvYignKi53aGwnKV06CiAgICBuYW1lLCB2ZXIgPSB3Lm5hbWUuc3BsaXQoJy0nKVs6Ml0KICAgIFdIRUVMUy5zZXRkZWZhdWx0KG5vcm0obmFtZSksIFtdKS5hcHBlbmQoKFZlcnNpb24odmVyKSwgdykpCgpPV04gPSB7J2Zhc3RhcGkvZmFzdGFwaSc6ICdmYXN0YXBpJywgJ1RleHR1YWxpemUvcmljaCc6ICdyaWNoJywKICAgICAgICdwc2YvcmVxdWVzdHMnOiAncmVxdWVzdHMnLCAnZW5jb2RlL2h0dHB4JzogJ2h0dHB4J30KT05MWV9GT1IgPSB7J3B5dGVzdF9odHRwYmluJzogeydwc2YvcmVxdWVzdHMnfSwgJ2h0dHBiaW4nOiB7J3BzZi9yZXF1ZXN0cyd9LAogICAgICAgICAgICAnZmxhc2dnZXInOiB7J3BzZi9yZXF1ZXN0cyd9LCAnYnJvdGxpY2ZmaSc6IHsncHNmL3JlcXVlc3RzJywgJ2VuY29kZS9odHRweCd9LAogICAgICAgICAgICAnYnJvdGxpJzogeydlbmNvZGUvaHR0cHgnfSwgJ3pzdGFuZGFyZCc6IHsnZW5jb2RlL2h0dHB4J319CgoKZGVmIHB5dGhvbl9mb3IodGFnOiBzdHIpIC0+IFBhdGg6CiAgICBpZiB0YWcgPT0gJ2NwMzEzJzoKICAgICAgICByZXR1cm4gUGF0aChzeXMuZXhlY3V0YWJsZSkKICAgIGRlc3QgPSBSVU4gLyBmJ3B5e3RhZ30nCiAgICB2ZXIgPSBmJ3t0YWdbMl19Lnt0YWdbMzpdfScKICAgIHB5ID0gZGVzdCAvICdweXRob24nIC8gJ2JpbicgLyBmJ3B5dGhvbnt2ZXJ9JwogICAgaWYgbm90IHB5LmV4aXN0cygpOgogICAgICAgIGRlc3QubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIHdpdGggdGFyZmlsZS5vcGVuKFBZRFMgLyBmJ2NweXRob24te3Zlcn0tbGludXgudGd6LmJpbicsICdyOmd6JykgYXMgdDoKICAgICAgICAgICAgdC5leHRyYWN0YWxsKGRlc3QpCiAgICByZXR1cm4gcHkKCgpDQUxJQlJBVElPTiA9IHsKICAgICdyaWNoXzMwNjQnOiB7J3B5Z21lbnRzJzogJzIuMTUuMSd9LCAncmljaF8zMTMwJzogeydweWdtZW50cyc6ICcyLjE1LjEnfSwKICAgICdyaWNoXzMyOTYnOiB7J3B5Z21lbnRzJzogJzIuMTYuMSd9LCAncmljaF8zNDY5JzogeydweWdtZW50cyc6ICcyLjE4LjAnfSwKICAgICdyaWNoXzM3ODInOiB7J3B5Z21lbnRzJzogJzIuMTkuMid9LCAncmljaF80MDcwJzogeydweWdtZW50cyc6ICcyLjE5LjInfSwKICAgICdyaWNoXzQwNzknOiB7J3B5Z21lbnRzJzogJzIuMjAuMCd9LAogICAgJ3JlcXVlc3RzXzY1ODknOiB7J3dlcmt6ZXVnJzogJzIuMy44JywgJ2ZsYXNrJzogJzIuMy4zJ30sCiAgICAncmVxdWVzdHNfNjU5Mic6IHsnd2Vya3pldWcnOiAnMi4zLjgnLCAnZmxhc2snOiAnMi4zLjMnfSwKICAgICdyZXF1ZXN0c182NjI5Jzogeyd3ZXJremV1Zyc6ICcyLjMuOCcsICdmbGFzayc6ICcyLjMuMyd9LAp9ClRBU0tfUFlUSE9OID0geydyZXF1ZXN0c182NzU3JzogJ2NwMzEyJywgJ3JpY2hfMzQ3Mic6ICdjcDMxMid9CkFTWU5DX1NISU0gPSB7J2Zhc3RhcGlfMTU3NDUnLCAnZmFzdGFwaV8xNTc2MycsICdmYXN0YXBpXzE1Nzg1J30KTk9fQ0VSVF9USU1FID0geydyZXF1ZXN0c182NzU3J30KCkJBU0UgPSB7fQpTSElNID0gUlVOIC8gJ3Jlc3VsdHMnIC8gJ3NoaW1zJwoKCmRlZiBiYXNlX2ZvcihyZXBvOiBzdHIsIHRhZzogc3RyKSAtPiBQYXRoOgogICAgaWYgKHJlcG8sIHRhZykgbm90IGluIEJBU0U6CiAgICAgICAgc2VsID0ge30KICAgICAgICBmb3IgbmFtZSwgb3B0cyBpbiBXSEVFTFMuaXRlbXMoKToKICAgICAgICAgICAgb3B0cyA9IFsodiwgdykgZm9yIHYsIHcgaW4gb3B0cyBpZiBmaXRzKHN0cih3KSwgdGFnKV0KICAgICAgICAgICAgaWYgbm90IG9wdHMgb3IgbmFtZSA9PSBPV05bcmVwb10gb3IgKG5hbWUgaW4gT05MWV9GT1IgYW5kIHJlcG8gbm90IGluIE9OTFlfRk9SW25hbWVdKToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGNvbXAgPSBbKHYsIHcpIGZvciB2LCB3IGluIG9wdHMgaWYgdy5wYXJlbnQgPT0gQ09NUCAvICd3aGVlbHMnXQogICAgICAgICAgICBpZiBuYW1lIGluICgncHl0ZXN0JywgJ3B5dGVzdF90aW1lb3V0Jyk6CiAgICAgICAgICAgICAgICBzZWxbbmFtZV0gPSBuZXh0KHcgZm9yIHYsIHcgaW4gb3B0cyBpZiBzdHIodikgaW4gKCc4LjQuMicsICcyLjEuMCcpKQogICAgICAgICAgICBlbGlmIGNvbXA6CiAgICAgICAgICAgICAgICBzZWxbbmFtZV0gPSBtYXgoY29tcClbMV0KICAgICAgICAgICAgZWxpZiBuYW1lICE9ICdweWdtZW50cyc6CiAgICAgICAgICAgICAgICBzZWxbbmFtZV0gPSBtYXgob3B0cylbMV0KICAgICAgICBkID0gUlVOIC8gJ3Jlc3VsdHMnIC8gZidzY29yZXJfc2l0ZV97T1dOW3JlcG9dfV97dGFnfScKICAgICAgICBzaHV0aWwucm10cmVlKGQsIGlnbm9yZV9lcnJvcnM9VHJ1ZSkKICAgICAgICBkLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBzdWJwcm9jZXNzLnJ1bihbc3RyKHB5dGhvbl9mb3IodGFnKSksICctbScsICdwaXAnLCAnaW5zdGFsbCcsICctcScsICctLW5vLWluZGV4JywKICAgICAgICAgICAgICAgICAgICAgICAgJy0tbm8tZGVwcycsICctLWlnbm9yZS1yZXF1aXJlcy1weXRob24nLCAnLS10YXJnZXQnLCBzdHIoZCksCiAgICAgICAgICAgICAgICAgICAgICAgICptYXAoc3RyLCBzZWwudmFsdWVzKCkpXSwgY2hlY2s9VHJ1ZSkKICAgICAgICBmb3Igc2hpbSBpbiAoJ2ltcC5weScsICd0ZWxuZXRsaWIucHknKToKICAgICAgICAgICAgc2h1dGlsLmNvcHkoQ09NUCAvICdkb2NrZXInIC8gc2hpbSwgZCAvIHNoaW0pCiAgICAgICAgQkFTRVsocmVwbywgdGFnKV0gPSBkCiAgICByZXR1cm4gQkFTRVsocmVwbywgdGFnKV0KCgpkZWYgcG9ldHJ5X3RvX3BlcDQ0MChzcGVjKToKICAgIHNwZWMgPSBzcGVjIGlmIGlzaW5zdGFuY2Uoc3BlYywgc3RyKSBlbHNlIHNwZWMuZ2V0KCd2ZXJzaW9uJywgJycpCiAgICBtID0gcmUuZnVsbG1hdGNoKHInXF4oXGQrKShcLlxkKyk/KFwuXGQrKT8nLCBzcGVjLnN0cmlwKCkpCiAgICByZXR1cm4gZic+PXtzcGVjLnN0cmlwKClbMTpdfSw8e2ludChtLmdyb3VwKDEpKSArIDF9JyBpZiBtIGVsc2UgKCcnIGlmIHNwZWMuc3RyaXAoKSA9PSAnKicgZWxzZSBzcGVjKQoKCkNVUlJFTlQgPSB7J2lkJzogTm9uZX0KCgpkZWYgc2V0dXBfZW52KGRvY2tlciwgc2FuZGJveF9pZCwgcmVwbz0nJywgKiprd2FyZ3MpOgogICAgdGlkID0gQ1VSUkVOVFsnaWQnXQogICAgdGFnID0gVEFTS19QWVRIT04uZ2V0KHRpZCwgJ2NwMzEzJykKICAgIHNpdGUgPSBQYXRoKGRvY2tlci5leGVjKHNhbmRib3hfaWQsICdweXRob24gLWMgImltcG9ydCBzaXRlOyBwcmludChzaXRlLmdldHNpdGVwYWNrYWdlcygpWzBdKSInKS5zdGRvdXQuc3RyaXAoKSkKICAgIGlmIHRhZyAhPSAnY3AzMTMnOgogICAgICAgIHZlbnZfZGlyID0gc2l0ZS5wYXJlbnRzWzJdCiAgICAgICAgc3VicHJvY2Vzcy5ydW4oW3N0cihweXRob25fZm9yKHRhZykpLCAnLW0nLCAndmVudicsICctLWNsZWFyJywgJy0td2l0aG91dC1waXAnLCBzdHIodmVudl9kaXIpXSwgY2hlY2s9VHJ1ZSkKICAgICAgICBzaXRlID0gbmV4dCh2ZW52X2Rpci5nbG9iKCdsaWIvcHl0aG9uMyovc2l0ZS1wYWNrYWdlcycpKQogICAgd3MgPSBkb2NrZXIuZXhlYyhzYW5kYm94X2lkLCAncHdkJykuc3Rkb3V0LnN0cmlwKCkKICAgIGJhc2UgPSBiYXNlX2ZvcihyZXBvLCB0YWcpCiAgICAoc2l0ZSAvICcwMF9zcmMucHRoJykud3JpdGVfdGV4dChmJ3t3c30vc3JjXG4nKQogICAgKHNpdGUgLyAnMDFfc2NvcmVyX2Jhc2UucHRoJykud3JpdGVfdGV4dChmJ3tiYXNlfVxuJykKICAgIChzaXRlIC8gJ3p6X3dvcmtzcGFjZV9wYXRocy5wdGgnKS53cml0ZV90ZXh0KAogICAgICAgICdcbicuam9pbihzdHIocCkgZm9yIHAgaW4gUGF0aCh3cykuaXRlcmRpcigpIGlmIHAuaXNfZGlyKCkgYW5kIG5vdCBwLm5hbWUuc3RhcnRzd2l0aCgnLicpKSArICdcbicpCiAgICBpZiB0aWQgaW4gQVNZTkNfU0hJTToKICAgICAgICAoU0hJTSkubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIF93cml0ZV9hc3luY19zaGltKFNISU0pCiAgICAgICAgKHNpdGUgLyAnMDJfYXN5bmNfc2hpbS5wdGgnKS53cml0ZV90ZXh0KGYne1NISU19XG4nKQogICAgaWYgdGlkIGluIE5PX0NFUlRfVElNRToKICAgICAgICAoc2l0ZSAvICcwM19ub19jZXJ0X3RpbWUucHRoJykud3JpdGVfdGV4dChmJ2ltcG9ydCBzeXM7IHN5cy5wYXRoLmluc2VydCgwLCB7c3RyKFNISU0pIXJ9KTsgaW1wb3J0IHN3ZV9ub19jZXJ0X3RpbWVcbicpCiAgICB0cnk6CiAgICAgICAgZGF0YSA9IHRvbWxsaWIubG9hZHMoUGF0aCh3cywgJ3B5cHJvamVjdC50b21sJykucmVhZF90ZXh0KCkpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIGRhdGEgPSB7fQogICAgcmVxcyA9IGxpc3QoZGF0YS5nZXQoJ3Byb2plY3QnLCB7fSkuZ2V0KCdkZXBlbmRlbmNpZXMnLCBbXSkpCiAgICByZXFzICs9IFtuICsgcG9ldHJ5X3RvX3BlcDQ0MChzKSBmb3IgbiwgcyBpbgogICAgICAgICAgICAgZGF0YS5nZXQoJ3Rvb2wnLCB7fSkuZ2V0KCdwb2V0cnknLCB7fSkuZ2V0KCdkZXBlbmRlbmNpZXMnLCB7fSkuaXRlbXMoKSBpZiBuICE9ICdweXRob24nXQogICAgcGlja3MsIHF1ZXVlLCBzZWVuID0ge30sIGxpc3QocmVxcyksIHNldCgpCiAgICB3aGlsZSBxdWV1ZToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJlcSA9IFJlcXVpcmVtZW50KHF1ZXVlLnBvcCgwKSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGtleSA9IG5vcm0ocmVxLm5hbWUpCiAgICAgICAgaWYga2V5IGluIHNlZW4gb3Iga2V5ID09IE9XTi5nZXQocmVwbykgb3IgKHJlcS5tYXJrZXIgYW5kIG5vdCByZXEubWFya2VyLmV2YWx1YXRlKHsnZXh0cmEnOiAnJ30pKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBzZWVuLmFkZChrZXkpCiAgICAgICAgb3B0aW9ucyA9IFsodiwgdykgZm9yIHYsIHcgaW4gV0hFRUxTLmdldChrZXksIFtdKQogICAgICAgICAgICAgICAgICAgaWYgZml0cyhzdHIodyksIHRhZykgYW5kIHJlcS5zcGVjaWZpZXIuY29udGFpbnModiwgcHJlcmVsZWFzZXM9VHJ1ZSkKICAgICAgICAgICAgICAgICAgIGFuZCBub3QgKGtleSBpbiBPTkxZX0ZPUiBhbmQgcmVwbyBub3QgaW4gT05MWV9GT1Jba2V5XSldCiAgICAgICAgaWYgb3B0aW9uczoKICAgICAgICAgICAgYmVzdCA9IG1heChvcHRpb25zKVsxXQogICAgICAgICAgICBwaWNrc1trZXldID0gYmVzdAogICAgICAgICAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShiZXN0KSBhcyB6OgogICAgICAgICAgICAgICAgbWV0YSA9IG5leHQobiBmb3IgbiBpbiB6Lm5hbWVsaXN0KCkgaWYgbi5lbmRzd2l0aCgnLmRpc3QtaW5mby9NRVRBREFUQScpKQogICAgICAgICAgICAgICAgcXVldWUgKz0gW2wuc3BsaXQoJzonLCAxKVsxXS5zdHJpcCgpIGZvciBsIGluIHoucmVhZChtZXRhKS5kZWNvZGUoKS5zcGxpdGxpbmVzKCkKICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBsLnN0YXJ0c3dpdGgoJ1JlcXVpcmVzLURpc3Q6JyldCiAgICBmb3IgbmFtZSwgdmVyIGluIENBTElCUkFUSU9OLmdldCh0aWQsIHt9KS5pdGVtcygpOgogICAgICAgIHBpY2tzW25hbWVdID0gbmV4dCh3IGZvciB2LCB3IGluIFdIRUVMU1tuYW1lXSBpZiBzdHIodikgPT0gdmVyIGFuZCBmaXRzKHN0cih3KSwgdGFnKSkKICAgIGlmIHBpY2tzOgogICAgICAgIHN1YnByb2Nlc3MucnVuKFtzdHIocHl0aG9uX2Zvcih0YWcpKSwgJy1tJywgJ3BpcCcsICdpbnN0YWxsJywgJy1xJywgJy0tbm8taW5kZXgnLCAnLS1uby1kZXBzJywKICAgICAgICAgICAgICAgICAgICAgICAgJy0tdXBncmFkZScsICctLWlnbm9yZS1yZXF1aXJlcy1weXRob24nLCAnLS10YXJnZXQnLCBzdHIoc2l0ZSksCiAgICAgICAgICAgICAgICAgICAgICAgICptYXAoc3RyLCBwaWNrcy52YWx1ZXMoKSldKQogICAgcmV0dXJuIHBpY2tzCgoKZGVmIF93cml0ZV9hc3luY19zaGltKHNoaW1fZGlyOiBQYXRoKToKICAgIGRpID0gc2hpbV9kaXIgLyAnc3dlX2FzeW5jX3NoaW0tMC4xLmRpc3QtaW5mbycKICAgIGRpLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIChkaSAvICdNRVRBREFUQScpLndyaXRlX3RleHQoJ01ldGFkYXRhLVZlcnNpb246IDIuMVxuTmFtZTogc3dlLWFzeW5jLXNoaW1cblZlcnNpb246IDAuMVxuJykKICAgIChkaSAvICdlbnRyeV9wb2ludHMudHh0Jykud3JpdGVfdGV4dCgnW3B5dGVzdDExXVxuc3dlX2FzeW5jX3NoaW0gPSBzd2VfYXN5bmNfc2hpbVxuJykKICAgIChzaGltX2RpciAvICdzd2VfYXN5bmNfc2hpbS5weScpLndyaXRlX3RleHQoJycnaW1wb3J0IGFzeW5jaW8sIGluc3BlY3QKaW1wb3J0IHB5dGVzdAoKCkBweXRlc3QuaG9va2ltcGwodHJ5Zmlyc3Q9VHJ1ZSkKZGVmIHB5dGVzdF9weWZ1bmNfY2FsbChweWZ1bmNpdGVtKToKICAgIGlmIGluc3BlY3QuaXNjb3JvdXRpbmVmdW5jdGlvbihweWZ1bmNpdGVtLm9iaik6CiAgICAgICAgYXJncyA9IHthOiBweWZ1bmNpdGVtLmZ1bmNhcmdzW2FdIGZvciBhIGluIHB5ZnVuY2l0ZW0uX2ZpeHR1cmVpbmZvLmFyZ25hbWVzfQogICAgICAgIGFzeW5jaW8ucnVuKHB5ZnVuY2l0ZW0ub2JqKCoqYXJncykpCiAgICAgICAgcmV0dXJuIFRydWUKICAgIHJldHVybiBOb25lCicnJykKICAgIChzaGltX2RpciAvICdzd2Vfbm9fY2VydF90aW1lLnB5Jykud3JpdGVfdGV4dCgnJydpbXBvcnQgc3NsCnNzbC5fc3NsLl90ZXN0X2RlY29kZV9jZXJ0ID0gbGFtYmRhICphLCAqKms6IE5vbmUKJycnKQoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgnLS10YXNrcycsIGRlZmF1bHQ9J2FsbCcpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tc2hhcmQnLCB0eXBlPWludCwgZGVmYXVsdD0wKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLW51bS1zaGFyZHMnLCB0eXBlPWludCwgZGVmYXVsdD0xKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLXJlc3VsdHMnLCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tbW9kZWwtdXJsJywgZGVmYXVsdD1vcy5lbnZpcm9uLmdldCgnTU9ERUxfVVJMJywgJ2h0dHA6Ly8xMjcuMC4wLjE6ODAwMC92MScpKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLW1vZGVsLW5hbWUnLCBkZWZhdWx0PW9zLmVudmlyb24uZ2V0KCdNT0RFTF9OQU1FJywgJ2dlbW1hLTQtMzFiLWl0LXFhdC13NGExNi1jdCcpKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLWFnZW50LWRpcicsIGRlZmF1bHQ9c3RyKFJVTiAvICdhZ2VudCcpKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIHRhc2tzID0gbG9hZF90YXNrcyhDT01QIC8gJ3Rhc2tzLmpzb25sJykKICAgIGlmIGFyZ3MudGFza3MgIT0gJ2FsbCc6CiAgICAgICAga2VlcCA9IHNldChhcmdzLnRhc2tzLnNwbGl0KCcsJykpCiAgICAgICAgdGFza3MgPSBbdCBmb3IgdCBpbiB0YXNrcyBpZiB0Lmluc3RhbmNlX2lkIGluIGtlZXBdCiAgICB0YXNrcyA9IFt0IGZvciBpLCB0IGluIGVudW1lcmF0ZSh0YXNrcykgaWYgaSAlIGFyZ3MubnVtX3NoYXJkcyA9PSBhcmdzLnNoYXJkXQogICAgcHJpbnQoZid7bGVuKHRhc2tzKX0gdGFza3MnLCBmbHVzaD1UcnVlKQoKICAgIHJlc3VsdHNfZGlyID0gUGF0aChhcmdzLnJlc3VsdHMgb3IgUlVOIC8gJ3Jlc3VsdHMnIC8gZidydW5fc2hhcmR7YXJncy5zaGFyZH1vZnthcmdzLm51bV9zaGFyZHN9JykKICAgIHJlc3VsdHNfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBtb2RlbHMgPSBNb2RlbFJlZ2lzdHJ5KCkKICAgIG1vZGVscy5yZWdpc3RlcihhcmdzLm1vZGVsX25hbWUsIExpdGVMbG0oCiAgICAgICAgbW9kZWw9ZiJvcGVuYWkve2FyZ3MubW9kZWxfbmFtZX0iLCBhcGlfYmFzZT1hcmdzLm1vZGVsX3VybCwgYXBpX2tleT0iRU1QVFkiKSkKICAgIGxpbWl0cywgZ2VuX2NvbnN0cmFpbnRzID0gYnVpbGRfc3VibWlzc2lvbl9saW1pdHMoKQoKICAgIGNmZyA9IEV2YWxDb25maWcoCiAgICAgICAgdGFza3NfcGF0aD1DT01QIC8gJ3Rhc2tzLmpzb25sJywgc25hcHNob3RzX2Rpcj1DT01QIC8gJ3NuYXBzaG90cycsCiAgICAgICAgcmVzdWx0c19kaXI9cmVzdWx0c19kaXIsIHN1Ym1pc3Npb25fZGlyPVBhdGgoYXJncy5hZ2VudF9kaXIpLCBtb2RlbHM9bW9kZWxzLAogICAgICAgIHNhbmRib3g9J3N1YnByb2Nlc3MnLCB0aW1lb3V0X3NlY29uZHM9MjQwLCB2ZXJpZmljYXRpb25fdGltZW91dF9zZWNvbmRzPTMwMCwKICAgICAgICBtYXhfdGltZV9taW51dGVzPTcsIG1heF90b29sX2NhbGxzPTI4LCBtYXhfdHVybnM9ODAsIGxpbWl0cz1saW1pdHMsCiAgICAgICAgZ2VuZXJhdGlvbl9jb25zdHJhaW50cz1nZW5fY29uc3RyYWludHMsIGFkYXB0ZXJfbWFuaWZlc3Q9Tm9uZSwKICAgICAgICBjb250ZXh0X2NhY2hlX2NvbmZpZz1Db250ZXh0Q2FjaGVDb25maWcobWluX3Rva2Vucz0yMDQ4LCB0dGxfc2Vjb25kcz0xODAwLCBjYWNoZV9pbnRlcnZhbHM9MTApLAogICAgICAgIGV2ZW50c19jb21wYWN0aW9uX2NvbmZpZz1FdmVudHNDb21wYWN0aW9uQ29uZmlnKAogICAgICAgICAgICBjb21wYWN0aW9uX2ludGVydmFsPTE1LCBvdmVybGFwX3NpemU9MiwgdG9rZW5fdGhyZXNob2xkPTE0MzM2LCBldmVudF9yZXRlbnRpb25fc2l6ZT01KSwKICAgICAgICBncmFwaF9kaXI9c3RyKENPTVAgLyAnZ3JhcGhzJyksIGVtYmVkZGluZ3NfZGlyPXN0cihDT01QIC8gJ2VtYmVkZGluZ3MnKSwKICAgICAgICB3aGVlbHNfZGlyPUNPTVAgLyAnd2hlZWxzJywgdmVyYm9zZT1GYWxzZSwgZGlzcGxheV9tb2RlPSdxdWlldCcpCiAgICBldiA9IEV2YWx1YXRvcihjZmcpCiAgICBldi5zYW5kYm94ID0gZXYuZG9ja2VyID0gU3VicHJvY2Vzc01hbmFnZXIodGltZW91dF9zZWNvbmRzPTI0MCwgc3lzdGVtX3NpdGVfcGFja2FnZXM9RmFsc2UpCiAgICB2ZXJpZmljYXRpb24uaW5zdGFsbF90ZXN0X2RlcGVuZGVuY2llcyA9IHNldHVwX2VudgogICAgYWdlbnRfcnVubmVyLmluc3RhbGxfdGVzdF9kZXBlbmRlbmNpZXMgPSBzZXR1cF9lbnYKCiAgICBwb29sID0gVGhyZWFkUG9vbEV4ZWN1dG9yKG1heF93b3JrZXJzPTEpCiAgICBmb3IgaSwgdGFzayBpbiBlbnVtZXJhdGUodGFza3MsIDEpOgogICAgICAgIENVUlJFTlRbJ2lkJ10gPSB0YXNrLmluc3RhbmNlX2lkCiAgICAgICAgdDAgPSB0aW1lLnRpbWUoKQogICAgICAgIHRyeToKICAgICAgICAgICAgcmVzID0gcG9vbC5zdWJtaXQoYXN5bmNpby5ydW4sIGV2LmV2YWx1YXRlX3Rhc2sodGFzaz10YXNrLCB0YXNrX2luZGV4PWksIHRvdGFsX3Rhc2tzPWxlbih0YXNrcykpKS5yZXN1bHQoKQogICAgICAgICAgICByb3cgPSBkaWN0KGlkPXRhc2suaW5zdGFuY2VfaWQsIHJlcG89dGFzay5yZXBvLCByZXNvbHZlZD1ib29sKHJlcy5yZXNvbHZlZCksCiAgICAgICAgICAgICAgICAgICAgICAgZXhpdF9jb2RlPXJlcy50ZXN0X2V4aXRfY29kZSwgdG9vbF9jYWxscz1yZXMudG9vbF9jYWxscywKICAgICAgICAgICAgICAgICAgICAgICBzZWNvbmRzPXJvdW5kKHRpbWUudGltZSgpIC0gdDAsIDEpLCBlcnJvcj0ocmVzLmVycm9yIG9yICcnKVs6MjAwXSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6ICAjIG5vcWE6IEJMRTAwMQogICAgICAgICAgICByb3cgPSBkaWN0KGlkPXRhc2suaW5zdGFuY2VfaWQsIHJlcG89dGFzay5yZXBvLCByZXNvbHZlZD1GYWxzZSwgZXhpdF9jb2RlPS0xLAogICAgICAgICAgICAgICAgICAgICAgIHRvb2xfY2FsbHM9MCwgc2Vjb25kcz1yb3VuZCh0aW1lLnRpbWUoKSAtIHQwLCAxKSwgZXJyb3I9cmVwcihlKVs6MjAwXSkKICAgICAgICB3aXRoIG9wZW4ocmVzdWx0c19kaXIgLyAncHJvZ3Jlc3MuanNvbmwnLCAnYScpIGFzIGZoOgogICAgICAgICAgICBmaC53cml0ZShqc29uLmR1bXBzKHJvdykgKyAnXG4nKQogICAgICAgIHByaW50KGpzb24uZHVtcHMocm93KSwgZmx1c2g9VHJ1ZSkKICAgIHByaW50KCdFVkFMIERPTkUnLCBmbHVzaD1UcnVlKQoKCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICBtYWluKCkK
'''.splitlines())).decode()
(Path('/kaggle/working/eval_local_kaggle.py')).write_text(EVAL_SRC)
import ast as _a; _a.parse(EVAL_SRC); print('wrote eval_local_kaggle.py', len(EVAL_SRC), 'parses OK')


In [ ]:
import base64, json, shutil
from pathlib import Path
BUNDLES = json.loads(base64.b64decode(''.join(r'''
...
'''.splitlines())).decode())
for tag, files in BUNDLES.items():
    d = Path('/kaggle/working') / f'agent_{tag}'
    if d.exists():
        shutil.rmtree(d)
    for rel, text in files.items():
        p = d / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(text)
    print(tag, '->', sorted(files))


In [ ]:
import re
import litellm
litellm.drop_params = True
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{MODEL_NAME}/2')
assert MODEL_PATH.exists(), MODEL_PATH
AGENT_DIR = Path('/kaggle/working/agent_v1')
declared = discover_declared_models(str(AGENT_DIR))
assert list(declared) == [MODEL_NAME], declared
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
gpu_count = torch.cuda.device_count()
tp_size = 1 if gpu_count == 1 else (2 if gpu_count == 2 else 4)
vllm_cfg = VllmConfig(
    model=str(MODEL_PATH), port=8000, host='127.0.0.1',
    tool_call_parser='gemma4', reasoning_parser='gemma4',
    max_model_len=32768, dtype='bfloat16', gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True, enable_lora=False,
    scheduler_reserve_full_isl=False, tensor_parallel_size=tp_size,
    startup_timeout=60 * 30, extra_args=())
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print('server at', server_instance.base_url, 'tp', tp_size)


In [ ]:
import os, sys, time
from pathlib import Path
base = str(server_instance.base_url).rstrip('/')
os.environ['MODEL_URL'] = base if base.endswith('/v1') else base + '/v1'
os.environ['MODEL_NAME'] = MODEL_NAME
print('MODEL_URL', os.environ['MODEL_URL'])
sys.path.insert(0, '/kaggle/working')
import eval_local_kaggle as EL
DEVSET = r'''
fastapi_11355,fastapi_13537,fastapi_14077,fastapi_14266,fastapi_14349,fastapi_14479,httpx_3672,rich_3278,rich_3470,rich_3472,rich_3535,rich_3772,rich_3894,rich_3905,rich_4076,rich_4079
'''.strip()
print('devset:', len(DEVSET.split(',')), 'tasks')
for tag in ('v2', 'v1'):
    results = Path('/kaggle/working') / f'results_{tag}'
    results.mkdir(parents=True, exist_ok=True)
    (results / 'progress.jsonl').write_text('')
    sys.argv = ['eval_local.py', '--tasks', DEVSET, '--results', str(results),
                '--agent-dir', str(Path('/kaggle/working') / f'agent_{tag}')]
    t0 = time.time()
    EL.main()
    print(tag, 'done in', round(time.time() - t0), 's')


In [ ]:
import json
from pathlib import Path
rows = {}
for tag in ('v1', 'v2'):
    rows[tag] = [json.loads(l) for l in (Path(f'/kaggle/working/results_{tag}/progress.jsonl')).read_text().splitlines() if l.strip()]
def summ(rows):
    r = [x for x in rows if x.get('resolved')]
    return len(r), len(rows)
a, na = summ(rows['v1']); b, nb = summ(rows['v2'])
w1 = sum(1 for x in rows['v1'] if x['resolved'] and not next(y for y in rows['v2'] if y['id'] == x['id'])['resolved'])
w2 = sum(1 for x in rows['v2'] if x['resolved'] and not next(y for y in rows['v1'] if y['id'] == x['id'])['resolved'])
print(f'v1: {a}/{na} resolved   v2: {b}/{nb} resolved   v1-only wins {w1}  v2-only wins {w2}')
comp = []
for x in rows['v1']:
    y = next((q for q in rows['v2'] if q['id'] == x['id']), None)
    comp.append({'id': x['id'], 'repo': x['repo'], 'v1': int(x['resolved']),
                 'v2': int(y['resolved']) if y else -1,
                 'v1_calls': x.get('tool_calls'), 'v2_calls': y.get('tool_calls') if y else None,
                 'v1_s': x.get('seconds'), 'v2_s': y.get('seconds') if y else None})
import csv, io
buf = io.StringIO()
w = csv.DictWriter(buf, fieldnames=list(comp[0].keys()))
w.writeheader(); w.writerows(comp)
out = Path('/kaggle/working/ab_summary.csv')
out.write_text(buf.getvalue())
print(out.read_text())
